In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
# 행(row) 다 보기
pd.set_option('display.max_rows', None)

# 열(column) 다 보기
pd.set_option('display.max_columns', None)

### 1. 대표 ETF을 통한 섹터 평가

In [28]:
# 섹터 ETF 예시 (미국)
sector_etfs = {
    "XLK": "Technology",            # 소프트웨어, 반도체, it서비스 예)Apple, Microsoft, Nvidia
    "XLC": "Communication Service", # 미디어, 인터넷 플랫폼, 통신 예)Google, Meta, Netflix
    "XLI": "Industrials",           # 제조, 운송, 항공, 방산
    "XLB": "Materials",             # 화학, 금속, 철강, 원자재
    "XLF": "Financials",            # 은행, 보험, 증권
    "XLE": "Energy",
    "XLV": "HealthCare",            # 제약, 바이오, 의료기기
    "XLU": "Utilities",             # 전기, 가스, 수도
    "XLRE" : "Real Estate",         # 리츠, 상업, 주거, 부동산
    "XLY": "ConsumerDiscretionary", # 필수소비재 (경기와 상관없이 꼭 사는 것들)
    "XLP": "ConsumerStaples"        # 선택소비재 (경기가 좋을 때 더 쓰는 것들)
}

market_etf = "SPY"

tickers = list(sector_etfs.keys()) + [market_etf]

data = yf.download(
    tickers,
    period="1y",
    auto_adjust=True,
    progress=False
)

In [29]:
# attention score 설계 철학
# 단기(20일) -> 관심 / 참여 / 변동성 폭발
# 종기(60일) -> 가격에 반영된 추세
# 전체(1년)  -> 정상 수준과 비고해 과도/저조 판단
def compute_features(price, volume, market_return):
    # 1. Volume Surge
    # 왜 최근 20일? 20일 기준은 약 1개월 거래일 기준
    # "단기적 집중 관심이 어느 정도 유지되는가"를 보여줌. 즉, 최근 1-2주 내 집중적인 참여를 포착하는 기간
    vol_surge = (
        np.round(volume.tail(20).mean() /
        volume.mean(), 2)
    )

    # 2. Trend Strength (60일 수익률)
    # 60일은 약 3개월 시점 -> 단기 관심이 가격에 어느 정도 반영되었는지 측정할 수 있는 중기 추세
    trend_strength = np.round(price.iloc[-1] / price.iloc[-60] - 1, 2)

    # 3. Volatility Expansion
    # 변동성이 크다는 말은 가격 움직임이 평소보다 활발
    # 시장 참여자들의 의견 충돌 / 불확실성 증가, 관심이 늘어나면서 거래 활동과 가격 변동이 함께 커진 상황
    returns = price.pct_change().dropna()
    vol_expansion = (
        np.round(returns.tail(20).std() /
        returns.std(), 2)
    )

    # 4. Relative Strength (시장 비교) 관심의 선택성 신호
    rel_strength = np.round(trend_strength - market_return, 2)

    return pd.Series({
        "volume_surge": vol_surge,
        "trend_strength": trend_strength,
        "vol_expansion": vol_expansion,
        "relative_strength": rel_strength
    })

In [30]:
features = []

# 시장 수익률 (SPY 기준)
market_price = data["Close"][market_etf]
market_return = market_price.iloc[-1] / market_price.iloc[-60] - 1

for ticker in sector_etfs.keys():
    price = data["Close"][ticker]
    volume = data["Volume"][ticker]

    f = compute_features(price, volume, market_return)
    f["ticker"] = ticker
    f["sector"] = sector_etfs[ticker]

    features.append(f)

df = pd.DataFrame(features).set_index("ticker")

In [31]:
score_cols = [
    "volume_surge",
    "trend_strength",
    "vol_expansion",
    "relative_strength"
]

df_z = df[score_cols].apply(
    lambda x: np.round((x - x.mean()) / x.std(), 2)
)

df["attention_score"] = df_z.sum(axis=1)

In [32]:
# volume_surge => 참여자 증가 (보통 1 이상이면 거래량 증가)
# vol_expansion => 변동성 확대 (보통 1 이상이면 변동성 확대)
# trend_strength => 가격상승 (상대적 가격비교)
# Relative_strength => 시장 대비 초과 상승 (상대적 가격비교)
df_result = (
    df[["sector", "attention_score"] + score_cols]
    .sort_values("attention_score", ascending=False)
)

df_result

,sector,attention_score,volume_surge,trend_strength,vol_expansion,relative_strength
ticker,,,,,,
XLV,HealthCare,8.89,1.22,0.15,1.00,0.12
XLK,Technology,1.21,1.16,0.02,0.80,-0.01
XLI,Industrials,0.80,1.08,0.02,0.85,-0.01
XLP,ConsumerStaples,0.51,1.09,0.02,0.80,-0.01
XLY,ConsumerDiscretionary,-0.12,1.04,0.04,0.67,0.01
XLB,Materials,-0.32,0.93,0.03,0.84,0.00
XLU,Utilities,-0.82,0.92,0.01,0.90,-0.02
XLC,Communication Service,-2.02,1.06,-0.00,0.62,-0.03
XLF,Financials,-2.07,0.96,0.02,0.63,-0.01


### 2. 개별주를 통한 섹터 평가
- participation : 얼마나 많은 종목이 움직였나 (참여도)
- breadth : 상승/활동이 넓게 퍼졌냐 (확산)
- vol_concentration : 거래가 소수 종목에 쏠렸냐 (쏠림, 높을수록 마이너스)
- dispersion : 종목간 움직임이 다양한가 (차별화)
- attention_score : 많은 종목이, 넓게, 특정 몇 개에만 쏠리지 않고, 각자 개별 스토리로 움직인다

In [63]:
def stock_singals(price, volume):
    vol_signal = (
        volume.tail(20).mean() > volume.mean() * 1.5
    )

    ret_60 = price.iloc[-1] / price.iloc[-60] - 1
    trend_signal = ret_60 > 0

    volat = price.pct_change().std()
    volat_recent = price.pct_change().tail(20).std()
    vol_exp_signal = volat_recent > volat

    is_judge = vol_signal & trend_signal & vol_exp_signal
    return is_judge, ret_60


In [67]:
results = []
signals, rets, vols = [], [], []
dict_us_stocks = pd.read_pickle('yf_chunk_all.pkl')
for ticker in dict_us_stocks.keys():

    history = dict_us_stocks[ticker]['history']
    info = dict_us_stocks[ticker]['info']
    if 'sector' not in info.keys(): continue
    if 'industry' not in info.keys(): continue
    sector = info['sector']
    industry = info['industry']
    if len(history) < 100 : continue

    sig, r = stock_singals(history['Close'], history['Volume'])
    signals.append(sig)
    rets.append(r)
    vols.append(history['Volume'].sum())

    participation = np.mean(signals)
    breadth = np.mean(np.array(rets) > 0)
    vol_concentration = (
        np.sum(sorted(vols, reverse=True)[:int(len(vols) * 0.1)])  / np.sum(vols)
    )
    dispersion = np.std(rets)

    results.append({
        "ticker": ticker,
        "sector": sector,
        "industry": industry,
        "participation": np.round(participation, 2),
        "breadth": np.round(breadth, 2),
        "vol_concentration": np.round(vol_concentration, 2),
        "dispersion": np.round(dispersion, 2)
    })

df_sector = pd.DataFrame(results)

/var/folders/jk/1j1mgc7x11122bdp64mnjcf40000gp/T/ipykernel_44595/3509258894.py:9: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  volat = price.pct_change().std()
/var/folders/jk/1j1mgc7x11122bdp64mnjcf40000gp/T/ipykernel_44595/3509258894.py:10: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  volat_recent = price.pct_change().tail(20).std()


- 개별들 티커들

In [73]:
cols = ["participation", "breadth", "vol_concentration", "dispersion"]

df_z = df_sector[cols].apply(lambda x: (x - x.mean()) / x.std())
df_sector["attention_score"] = (np.round(
    df_z["participation"] + df_z["breadth"] - df_z["vol_concentration"] + df_z["dispersion"], 2)
)
df_sector.sort_values(by='attention_score', ascending=False).head(10)

,ticker,sector,industry,participation,breadth,vol_concentration,dispersion,attention_score
3,CGC,Healthcare,Drug Manufacturers - Specialty & Generic,0.50,1.00,0.00,0.25,55.36
4,PROK,Healthcare,Biotechnology,0.40,0.80,0.00,0.30,45.04
2,ANNX,Healthcare,Biotechnology,0.33,1.00,0.00,0.29,44.53
5,HSTM,Healthcare,Health Information Services,0.33,0.67,0.00,0.32,37.49
6,SMBC,Financial Services,Banks - Regional,0.29,0.71,0.00,0.29,34.42
7,AIOT,Technology,Software - Infrastructure,0.25,0.62,0.00,0.28,28.88
8,TREE,Financial Services,Financial Conglomerates,0.22,0.56,0.00,0.30,25.98
1,ACDC,Energy,Oil & Gas Equipment & Services,0.00,1.00,0.00,0.14,14.73
16,LYEL,Healthcare,Biotechnology,0.12,0.59,0.52,0.40,11.55
18,PRME,Healthcare,Biotechnology,0.11,0.53,0.46,0.39,10.31


-들 섹터들

In [82]:
cols = ["participation", "breadth", "vol_concentration", "dispersion"]
sector_score = (
    df_sector
    .groupby(["sector"])
    .agg(
        participation = ("participation", "mean"),
        breadth = ("breadth", "mean"),
        vol_concentration = ("vol_concentration", "mean"),
        dispersion = ("dispersion", "mean"),
        cnt = ("attention_score", "count"),
        attention_score = ("attention_score", "mean")
    )
    .query("cnt >= 30")
    .round(2)
    .sort_values(by="attention_score", ascending=False)
)

sector_score[sector_score["attention_score"] > 0]


,participation,breadth,vol_concentration,dispersion,cnt,attention_score
sector,,,,,,
Healthcare,0.04,0.49,0.67,0.41,1043,0.74
Technology,0.04,0.49,0.68,0.41,731,0.43
Communication Services,0.04,0.49,0.68,0.41,268,0.15
Consumer Defensive,0.04,0.48,0.68,0.42,230,0.02


In [83]:
df_sector_agg = df_sector.groupby(['sector']).size().reset_index(name='cnt').sort_values(by='cnt', ascending=False)
df_sector_agg

,sector,cnt
6,Financial Services,1361
7,Healthcare,1043
10,Technology,731
8,Industrials,640
3,Consumer Cyclical,546
9,Real Estate,276
2,Communication Services,268
4,Consumer Defensive,230
5,Energy,226
1,Basic Materials,222
